# LLM Agent Benchmarking - Argentina Teacher Salaries

This notebook analyzes the model evaluation runs from yesterday (**March 5, 2026**) for the `DataJournalistAgent`.

In [1]:
import mlflow
import pandas as pd
import plotly.express as px
import os
import json
from datetime import datetime, timedelta

# 1. Setup MLflow Connection
mlflow.set_tracking_uri("sqlite:///mlflow.db")
experiment_name = "Agent_Comparison_March_2026"
experiment = mlflow.get_experiment_by_name(experiment_name)

if not experiment:
    exps = mlflow.search_experiments()
    print(f"Experiment '{experiment_name}' not found.")
    print("Available experiments:", [e.name for e in exps])
    if exps:
        experiment = exps[-1]
        print(f"Defaulting to most recent experiment: {experiment.name} (ID: {experiment.experiment_id})")
else:
    print(f"Experiment ID: {experiment.experiment_id}")

Experiment ID: 1


In [3]:
# 2. Search Runs by day
today = datetime(2026, 3, 6)
start_timestamp = int(today.timestamp() * 1000)
end_timestamp = int((today + timedelta(days=1)).timestamp() * 1000)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"start_time >= {start_timestamp} AND start_time < {end_timestamp}"
)

if not runs.empty:
    # Sort by start_time descending to prefer latest runs if there are duplicates
    runs = runs.sort_values("start_time", ascending=False)
    print(f"Found {len(runs)} runs from March 6, 2026.")
    display(runs[['run_id', 'start_time', 'tags.mlflow.runName', 'params.model']].head())
else:
    print("No runs found for the specified period.")

Found 4 runs from March 6, 2026.


,run_id,start_time,tags.mlflow.runName,params.model
0,250f404c8569424895373afd878794f7,2026-03-06 16:32:38.807000+00:00,Model_qwen3.5:9b,ollama/qwen3.5:9b
1,29d72d8b82314756b85d432d6086631d,2026-03-06 16:32:04.944000+00:00,Model_gpt-4o-mini,openai/gpt-4o-mini
2,bd3cdabafd9145cd9191e3721b6f1b05,2026-03-06 16:29:39.277000+00:00,Model_gpt-4o-mini,openai/gpt-4o-mini
3,5723739f1d4047c7a034efbb1a9016fc,2026-03-06 16:26:39.572000+00:00,Model_gpt-4o-mini,openai/gpt-4o-mini


In [4]:
# 3. Extract and Aggregate Evaluation Results
all_results = []

for idx, row in runs.iterrows():
    run_id = row['run_id']
    model_name = row.get('params.model', 'Unknown')
    
    # Path to evaluation artifact
    artifact_path = os.path.join("mlruns", experiment.experiment_id, run_id, "artifacts", "eval_results.json")
    
    if not os.path.exists(artifact_path):
        # Try absolute path
        artifact_path = os.path.join(os.getcwd(), "mlruns", experiment.experiment_id, run_id, "artifacts", "eval_results.json")
    
    if os.path.exists(artifact_path):
        try:
            with open(artifact_path, 'r') as f:
                data = json.load(f)
                if 'data' in data and 'columns' in data:
                    df_run = pd.DataFrame(data['data'], columns=data['columns'])
                else:
                    df_run = pd.DataFrame(data)
                
                df_run['run_id'] = run_id
                df_run['model'] = model_name
                # Add a timestamp from the run for later deduplication if needed
                df_run['run_start_time'] = row['start_time']
                all_results.append(df_run)
        except Exception as e:
            print(f"Error reading artifact for run {run_id}: {e}")

if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    # Deduplicate: if same model answered same question multiple times, keep the latest run's answer
    results_df = results_df.sort_values('run_start_time', ascending=False).drop_duplicates(['question', 'model'])
    print(f"Loaded {len(results_df)} unique evaluation cases.")
else:
    print("No evaluation results found in artifacts.")
    results_df = pd.DataFrame()

results_df.head()

Loaded 9 unique evaluation cases.


,question,answer,ground_truth,model,run_id,run_start_time
0,Which is the top paying province in September ...,"Neuquén: $1,471,200.25","Neuquén ($1,471,200)",ollama/qwen3.5:9b,250f404c8569424895373afd878794f7,2026-03-06 16:32:38.807000+00:00
1,What was the purchasing power loss in Buenos A...,Agent stopped due to iteration limit or time l...,-30.45%,ollama/qwen3.5:9b,250f404c8569424895373afd878794f7,2026-03-06 16:32:38.807000+00:00
2,Which provinces lost more purchasing power bet...,"San Luis (-27.41%), La Pampa (-20.01%), Santia...","Neuquén (+23.8%), Santa Cruz (+21.3%), Tierra ...",ollama/qwen3.5:9b,250f404c8569424895373afd878794f7,2026-03-06 16:32:38.807000+00:00
3,Which is the top paying province in September ...,"Neuquén, 1471200.25","Neuquén ($1,471,200)",openai/gpt-4o-mini,29d72d8b82314756b85d432d6086631d,2026-03-06 16:32:04.944000+00:00
4,What was the purchasing power loss in Buenos A...,-30.45%,-30.45%,openai/gpt-4o-mini,29d72d8b82314756b85d432d6086631d,2026-03-06 16:32:04.944000+00:00


## Visualizing Comparisons

We compare the answers provided by different models for the same questions.

In [7]:
# Pivot the results for direct comparison using pivot_table to be extra safe
if not results_df.empty:
    # Use pivot_table with 'first' as aggregator to handle any remaining duplicates gracefully
    comparison_df = results_df.pivot_table(
        index=['question', 'ground_truth'], 
        columns='model', 
        values='answer', 
        aggfunc='first'
    ).reset_index()
    
    # Display the comparison
    pd.set_option('display.max_colwidth', None)
    display(comparison_df.style.set_properties(**{'text-align': 'left', 'vertical-align': 'top'}))
else:
    print("Results dataframe is empty.")

model,question,ground_truth,ollama/qwen3.5:9b,openai/gpt-4o-mini
0,Compare the salary evolution of Córdoba versus Inflation (IPC) between December 2023 and September 2025.,"Salary: +175.90%, IPC: +165.60% (Beat inflation by ~10 pts)",nan,"Nominal Salary: +175.90%, Inflation (IPC): +165.60%, Poverty Line (CBT): +137.37%, Net result vs Inflation: +10.30 points."
1,What was the purchasing power loss in Buenos Aires between September 2023 and September 2025?,-30.45%,Agent stopped due to iteration limit or time limit.,-30.45%
2,What was the real salary index for Santa Fe in June 2024 (Base 100 = September 2023)?,75.94,nan,99.685600
3,Which is the top paying province in September 2025 and what is the salary?,"Neuquén ($1,471,200)","Neuquén: $1,471,200.25","Neuquén, 1471200.25"
4,Which province had the highest percentage growth in real salary between September 2023 and September 2025?,Neuquén (+23.8%),nan,Neuquén: +23.78%
5,Which provinces lost more purchasing power between November 2023 and today?,"Neuquén (+23.8%), Santa Cruz (+21.3%), Tierra del Fuego (+9.7%) Salta (-34.5%), Santiago del Estero (-34.5%), San Luis (-47.3%)","San Luis (-27.41%), La Pampa (-20.01%), Santiago del Estero (-13.94%), Santa Fe (-8.51%), Ciudad de Buenos Aires (-3.63%)","San Luis: -48.86%, Mendoza: -33.69%, Buenos Aires: -32.77%"


In [6]:
# 4. Simple Accuracy Metric (Human-in-the-loop candidate)
import re

def simple_score(row):
    gt = str(row['ground_truth'])
    answer = str(row['answer'])
    # Extract numbers or key substrings
    numbers = re.findall(r"\d+(?:[.,]\d+)?", gt)
    if not numbers:
        return 1 if gt.lower() in answer.lower() else 0
    
    match_count = sum(1 for n in numbers if n in answer)
    return match_count / len(numbers) if numbers else 0

if not results_df.empty:
    results_df['rough_score'] = results_df.apply(simple_score, axis=1)
    
    summary = results_df.groupby('model')['rough_score'].mean().reset_index()
    fig = px.bar(summary, x='model', y='rough_score', title="Model Rough Accuracy (Keyword matching)",
                 labels={'rough_score': 'Avg Match Score'}, color='model', range_y=[0, 1])
    fig.show()